# Static Features
- [Static Features](#static-features)
- [Base Analysis](#base-analysis)
- [Dates](#check-dates)
- [Location](#location)
- [Flat Characteristics](#flat-characteristics)
- [Lease Characteristics](#lease-characteristics)
- [Clean Function](#clean-function)
- [Join Composite Keys](#join-composite-keys)

Written By: Keltonn [A0222591J]

AI Used: Claude Sonnet 5.5 for API Lookup

In this notebook, we will be joining all the static location based features for the model.

### Feature Types

There are two general types of features for this problem:

|Feature Type|Description|
|---|---|
| Temporal Features | Features that change every month, like stock market data and Economic Indicators |
| Static Features | Features that have no trend, don't change. Includes most location features, such as housing type |

### Feature Descriptions

For this notebook, we will be joining all of the created location based static features for the dataset. This includes the following:

|Feature Store|Description|
|---|---|
| Nearest Railway Stations | Number of nearby railway stations, distance to railway stations, number of interchanges, foot traffic etc. |
| School | Characteristics of the Nearest School |
| Regional Demographics | Number of people age over 65 etc. |
| Location Cluster | Output from our Lat Long clustering using DBScan |



In [85]:
import pandas as pd
import numpy as np
import scipy as scp
import math
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
# -- Paths --
ROOT = '/Users/keltonn/Desktop/Projects/housing_rent_price/final_report'

# -- Datasets -- 
TRAIN_HOUSE_PATH = f"{ROOT}/data/df_keys/train-house.csv"
TEST_HOUSE_PATH = f"{ROOT}/data/df_keys/test-house.csv"
STATION_KEYS_PATH = f"{ROOT}/data/features/raw/raw_graph_data/station-keys.csv"
MRT_EDGES_PATH = f"{ROOT}/data/features/raw/raw_graph_data/sg-mrt-edges-weighted.csv"
CLUSTER_OUTPUT_PATH = f"{ROOT}/data/features/cleaned/static_features/location_cluster"

# -- DBScan model --
DBSCAN_PATH = f'{ROOT}/01_data_processing/01_clustering_features/models'

# Project default distance is Haversine in km. sklearn's haversine metric returns
# distances in radians on the unit sphere, so multiply by the Earth radius to get km.
EARTH_RADIUS_KM = 6371.0088

# Approximate lat/long bounding box of mainland Singapore
SG_LAT_RANGE = (1.20, 1.48)
SG_LON_RANGE = (103.60, 104)

# Join all features

In [59]:
# Keys, train and test set
train_house = pd.read_csv(TRAIN_HOUSE_PATH)
test_house = pd.read_csv(TEST_HOUSE_PATH)
stat_key = pd.read_csv(f'{ROOT}/data/features/raw/location_data_mining/station-keys.csv')[[
    'NAME', 'nearest_mall', 'n_lines', 'status',
       'n_malls_500m', 'nearest_mall_m'
]]
stat_key.columns = ['nearest_station','nearest_mall','train_lines','status','n_malls_500m','nearest_mall_m']
stat_key['status'] = stat_key['status'].apply(lambda x: 1 if x == 'open' else 0)
stat_key.columns = ['nearest_station','nearest_mall','train_lines','station_is_open','n_malls_500m','nearest_mall_m']
closest_mrt_station = pd.read_csv(f'{ROOT}/data/features/raw/location_data_mining/hdb_nearest_station.csv')
closest_school = pd.read_csv(f'{ROOT}/data/features/raw/location_data_mining/hdb_nearest_school.csv')

# Train Traffic Features
traffic = pd.read_csv(f'{ROOT}/data/features/raw/location_data_mining/station-traffic-std.csv')
traffic.columns = ['nearest_station', 'mrt_traffic_tap_in_amount_norm',
       'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm',
       'mrt_traffic_weekday_tap_norm', 'mrt_traffic_weekend_tap_norm',
       'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm',
       'mrt_traffic_busiest_hour']

# Cluster Output
train_lap = pd.read_csv(f'{ROOT}/data/features/cleaned/static_features/location_cluster/train_dbscan_cluster.csv').drop(['LATITUDE','LONGITUDE','POSTAL_CODE'], axis = 1)
test_lap = pd.read_csv(f'{ROOT}/data/features/cleaned/static_features/location_cluster/test_dbscan_cluster.csv').drop(['LATITUDE','LONGITUDE','POSTAL_CODE'], axis = 1)

# Planning Area features
plan_area = pd.read_csv(f'{ROOT}/data/features/raw/collected_data/zoning_data/sg-demographics-planning-area.csv')[[
    'planning_area',
    'total_population', 
    'pct_age_0_14', 
    'pct_age_15_64', 
    'pct_age_65_plus',
    'unemployment_rate', 
    'total_households', 
    'pct_hdb',
    'pct_private_housing', 'pct_hh_income_ge_10k', 'pct_hh_income_below_2k',
    'avg_household_size', 'pct_chinese', 'pct_malay', 'pct_indian',
    'pct_owner_occupied', 'pct_pmet'
]]
plan_ft = []
for c in plan_area.columns:
    if c != 'planning_area':
        plan_ft.append(f'plan_area_ft_{c}')
    else:
        plan_ft.append('PLANNING_AREA')
plan_area.columns = plan_ft

# Subzone Area features
subzone = pd.read_csv(f'{ROOT}/data/features/raw/collected_data/zoning_data/sg-demographics-subzone.csv')[
    ['total_population', 'pct_age_0_14', 'pct_age_15_64', 'pct_age_65_plus', 'pct_male']
]

# Mall Features
mall = pd.read_csv(f'{ROOT}/data/features/raw/collected_data/mall_features/sg-mall-features-wikipedia.csv')


In [60]:
# Join Cluster
print(f'Train: {train_house.shape}')
print(f'Test: {test_house.shape}')
train = pd.merge(train_house, train_lap, how = 'left',on = ['TOWN', 'BLOCK','STREET',]).reset_index(drop=True)
test = pd.merge(test_house, test_lap, how = 'left',on = ['TOWN', 'BLOCK','STREET',]).reset_index(drop=True)

# Join Planning Area
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, plan_area, how = 'left',on = ['PLANNING_AREA'])
test = pd.merge(test, plan_area, how = 'left',on = ['PLANNING_AREA'])

# Join nearest_station
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, closest_mrt_station, how = 'left',on = ['TOWN', 'BLOCK','STREET'])
test = pd.merge(test, closest_mrt_station, how = 'left',on = ['TOWN', 'BLOCK','STREET'])

# Join nearest_school
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, closest_school, how = 'left',on = ['TOWN', 'BLOCK','STREET'])
test = pd.merge(test, closest_school, how = 'left',on = ['TOWN', 'BLOCK','STREET'])

# Join MRT Traffic
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, traffic, how = 'left',on = ['nearest_station'])
test = pd.merge(test, traffic, how = 'left',on = ['nearest_station'])

# Join Nearest Mall
print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
train = pd.merge(train, stat_key, how = 'left',on = ['nearest_station'])
test = pd.merge(test, stat_key, how = 'left',on = ['nearest_station'])

print(f'Train: {train.shape}')
print(f'Test: {test.shape}')

Train: (149999, 18)
Test: (50000, 17)
Train: (149999, 19)
Test: (50000, 18)
Train: (149999, 35)
Test: (50000, 34)
Train: (149999, 39)
Test: (50000, 38)
Train: (149999, 62)
Test: (50000, 61)
Train: (149999, 70)
Test: (50000, 69)
Train: (149999, 75)
Test: (50000, 74)


In [86]:
label = 'MONTHLY_RENT'

id_col = ['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH', 'LEASE_COMMENCE_DATE',
       'TOWN', 'BLOCK', 'STREET',
       'POSTAL_CODE', 'LATITUDE', 'LONGITUDE', 'YEAR_COMPLETED',
       'SUBZONE', 'PLANNING_AREA', 'REGION', 'nearest_station', 'nearest_mall',
       'nearest_school','nearest_pri_school', 'nearest_sec_school', 'nearest_jc_school', 
       ]

num_col = ['FLOOR_AREA_SQM','plan_area_ft_total_population',
       'plan_area_ft_pct_age_0_14', 'plan_area_ft_pct_age_15_64',
       'plan_area_ft_pct_age_65_plus', 'plan_area_ft_unemployment_rate',
       'plan_area_ft_total_households', 'plan_area_ft_pct_hdb',
       'plan_area_ft_pct_private_housing', 'plan_area_ft_pct_hh_income_ge_10k',
       'plan_area_ft_pct_hh_income_below_2k',
       'plan_area_ft_avg_household_size', 'plan_area_ft_pct_chinese',
       'plan_area_ft_pct_malay', 'plan_area_ft_pct_indian',
       'plan_area_ft_pct_owner_occupied', 'plan_area_ft_pct_pmet',
       'nearest_station_m', 'n_stations_500m',
       'n_stations_1000m', 'mrt_traffic_tap_in_amount_norm','n_malls_500m', 'nearest_mall_m',
       'nearest_school_m','nearest_pri_school_m', 
       'nearest_sec_school_m',
       'sec_sch_score_entry_min', 'sec_sch_score_entry_max',
       'nearest_jc_school_m',
       'jc_arts_entry_min', 'jc_arts_entry_max', 'jc_science_entry_min',
       'jc_science_entry_max',
       'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm',
       'mrt_traffic_weekday_tap_norm', 'mrt_traffic_weekend_tap_norm',
       'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm']

cat_col_num = ['MAX_FLOOR',
 'mrt_traffic_busiest_hour',
 'train_lines',
 'station_is_open']
cat_col = [i for i in test.columns if i not in num_col+id_col+[label]+cat_col_num]

In [62]:
train.to_csv(f'{ROOT}/data/features/cleaned/static_features/unprocessed/train_feature_raw.csv',index = False)
test.to_csv(f'{ROOT}/data/features/cleaned/static_features/unprocessed/test_feature_raw.csv',index = False)

# Null Analysis

Here we run a check on the feature join to see what features have missing values. The two feature types that have null values are the Railway station traffic features, which have missing values due to some stations not being open at the time of data collection, and the demographic features, which have null values due to the government data being incomplete.

In these cases, we fill the null values with -1, consistent with all our other static features. (The entry cutoff points for some schools are also filled with -1 if they are missing)

In [69]:
train = pd.read_csv(f'{ROOT}/data/features/cleaned/static_features/unprocessed/train_feature_raw.csv')
test = pd.read_csv(f'{ROOT}/data/features/cleaned/static_features/unprocessed/test_feature_raw.csv')

In [70]:
# Null rate of the model features
features = num_col + cat_col_num + cat_col

null_rate = pd.DataFrame({
    'train_null_pct': train[features].isna().mean() * 100,
    'test_null_pct': test[features].isna().mean() * 100,
    'train_null_count': train[features].isna().sum(),
    'test_null_count': test[features].isna().sum(),
}).round(2)

has_null = (null_rate['train_null_count'] > 0) | (null_rate['test_null_count'] > 0)
print(f'Train: {train.shape}, Test: {test.shape}')
print(f'{has_null.sum()} of {len(features)} features have nulls')
missing_df = null_rate[has_null].sort_values('train_null_pct', ascending=False).reset_index()
missing_df.columns = ['feature_name', 'train_null_pct', 'test_null_pct', 'train_null_count',
       'test_null_count']
display(missing_df)

Train: (149999, 75), Test: (50000, 74)
16 of 55 features have nulls


,feature_name,train_null_pct,test_null_pct,train_null_count,test_null_count
0,mrt_traffic_tap_in_amount_norm,15.38,15.90,23076,7948
1,mrt_traffic_tap_out_amount_norm,15.38,15.90,23076,7948
2,mrt_traffic_total_tap_norm,15.38,15.90,23076,7948
3,mrt_traffic_weekday_tap_norm,15.38,15.90,23076,7948
4,mrt_traffic_weekend_tap_norm,15.38,15.90,23076,7948
5,mrt_traffic_am_peak_norm,15.38,15.90,23076,7948
6,mrt_traffic_pm_peak_norm,15.38,15.90,23076,7948
7,mrt_traffic_busiest_hour,15.38,15.90,23076,7948
8,plan_area_ft_unemployment_rate,0.59,0.63,890,317
9,plan_area_ft_pct_hdb,0.59,0.63,890,317


In [71]:
# MRT traffic nulls by nearest station:
mrt_cols = [c for c in test.columns if c.startswith('mrt_traffic_')]
traffic_stations = set(traffic['nearest_station'])

def station_null_summary(df):
    g = (
        df.assign(mrt_null=df[mrt_cols].isna().any(axis=1))
        .groupby('nearest_station')
        .agg(n_rows=('mrt_null', 'size'), null_n=('mrt_null', 'sum'), station_is_open=('station_is_open', 'first'))
    )
    g['in_traffic_file'] = g.index.isin(traffic_stations)
    return g

train_stn = station_null_summary(train)
test_stn = station_null_summary(test)


# Stations with no traffic data, by number of rows affected
missing_stn = (
    train_stn[train_stn['null_n'] > 0][['n_rows', 'station_is_open']].rename(columns={'n_rows': 'missing_rows_train'})
    .join(test_stn[test_stn['null_n'] > 0][['n_rows']].rename(columns={'n_rows': 'missing_rows_test'}))
    .sort_values('missing_rows_train', ascending=False)
)[['missing_rows_train','missing_rows_test','station_is_open']]
missing_stn

,missing_rows_train,missing_rows_test,station_is_open
nearest_station,,,
serangoon north mrt station,2787,949,1
pasir ris east mrt station,1779,593,1
teck ghee mrt station,1740,596,1
toh guan mrt station,1612,556,1
choa chu kang west mrt station,1302,403,1
bahar junction mrt station,1252,427,1
west coast mrt station,1204,404,1
jurong west mrt station,1087,395,1
tavistock mrt station,888,298,1


In [72]:
plan_cols = [c for c in train.columns if c.startswith('plan_area_ft_')]
plan_null_cols = [c for c in plan_cols if train[c].isna().any() or test[c].isna().any()]
print(f'{len(plan_null_cols)} of {len(plan_cols)} plan area columns have nulls')

def plan_null_summary(df):
    g = (
        df.assign(plan_null=df[plan_null_cols].isna().any(axis=1))
        .groupby('PLANNING_AREA')
        .agg(n_rows=('plan_null', 'size'), null_n=('plan_null', 'sum'))
    )
    g['in_plan_area_file'] = g.index.isin(set(plan_area['PLANNING_AREA']))
    return g

train_pa = plan_null_summary(train)
test_pa = plan_null_summary(test)

# Null rate by month (train and test), to check for a time link
plan_null_by_month = (
    pd.concat([train.assign(split='train'), test.assign(split='test')])
    .assign(plan_null=lambda d: d[plan_null_cols].isna().any(axis=1))
    .groupby(['split', 'RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH'])['plan_null'].mean().mul(100).round(2)
)

# Source values for the null planning areas: the join matched, but the source columns are NaN
null_pa_names = train_pa[train_pa['null_n'] > 0].index
plan_area[plan_area['PLANNING_AREA'].isin(null_pa_names)].set_index('PLANNING_AREA').T

8 of 16 plan area columns have nulls


PLANNING_AREA,rochor,changi
plan_area_ft_total_population,26240.00,3700.00
plan_area_ft_pct_age_0_14,10.06,22.70
plan_area_ft_pct_age_15_64,67.38,68.38
plan_area_ft_pct_age_65_plus,22.68,10.27
plan_area_ft_unemployment_rate,NaN,NaN
plan_area_ft_total_households,0.00,0.00
plan_area_ft_pct_hdb,NaN,NaN
plan_area_ft_pct_private_housing,NaN,NaN
plan_area_ft_pct_hh_income_ge_10k,NaN,NaN
plan_area_ft_pct_hh_income_below_2k,NaN,NaN


In [73]:
# Fill null values
for ft in missing_df['feature_name'].unique().tolist():
    train[ft] = train[ft].fillna(-1)
    test[ft] = test[ft].fillna(-1)


# Check null rate again to make sure that all missing values are removed
null_rate = pd.DataFrame({
    'train_null_pct': train[features].isna().mean() * 100,
    'test_null_pct': test[features].isna().mean() * 100,
    'train_null_count': train[features].isna().sum(),
    'test_null_count': test[features].isna().sum(),
}).round(2)

has_null = (null_rate['train_null_count'] > 0) | (null_rate['test_null_count'] > 0)
print(f'Train: {train.shape}, Test: {test.shape}')
print(f'{has_null.sum()} of {len(features)} features have nulls')
missing_df = null_rate[has_null].sort_values('train_null_pct', ascending=False).reset_index()
missing_df.columns = ['feature_name', 'train_null_pct', 'test_null_pct', 'train_null_count',
       'test_null_count']
display(missing_df)

Train: (149999, 75), Test: (50000, 74)
0 of 55 features have nulls


,feature_name,train_null_pct,test_null_pct,train_null_count,test_null_count


In [74]:
train.to_csv(f'{ROOT}/data/features/cleaned/static_features/unprocessed/train_feature_raw.csv',index = False)
test.to_csv(f'{ROOT}/data/features/cleaned/static_features/unprocessed/test_feature_raw.csv',index = False)

# One Hot Encoding

We one hot encode the categorical features to generate the categorical columns

In [80]:
train = pd.read_csv(f'{ROOT}/data/features/cleaned/static_features/unprocessed/train_feature_raw.csv')
test = pd.read_csv(f'{ROOT}/data/features/cleaned/static_features/unprocessed/test_feature_raw.csv')

In [81]:
label = 'MONTHLY_RENT'

id_col = ['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH', 'LEASE_COMMENCE_DATE',
       'TOWN', 'BLOCK', 'STREET',
       'POSTAL_CODE', 'LATITUDE', 'LONGITUDE', 'YEAR_COMPLETED',
       'SUBZONE', 'PLANNING_AREA', 'REGION', 'nearest_station', 'nearest_mall',
       'nearest_school','nearest_pri_school', 'nearest_sec_school', 'nearest_jc_school', 
       ]

num_col = ['FLOOR_AREA_SQM','plan_area_ft_total_population',
       'plan_area_ft_pct_age_0_14', 'plan_area_ft_pct_age_15_64',
       'plan_area_ft_pct_age_65_plus', 'plan_area_ft_unemployment_rate',
       'plan_area_ft_total_households', 'plan_area_ft_pct_hdb',
       'plan_area_ft_pct_private_housing', 'plan_area_ft_pct_hh_income_ge_10k',
       'plan_area_ft_pct_hh_income_below_2k',
       'plan_area_ft_avg_household_size', 'plan_area_ft_pct_chinese',
       'plan_area_ft_pct_malay', 'plan_area_ft_pct_indian',
       'plan_area_ft_pct_owner_occupied', 'plan_area_ft_pct_pmet',
       'nearest_station_m', 'n_stations_500m',
       'n_stations_1000m', 'mrt_traffic_tap_in_amount_norm','n_malls_500m', 'nearest_mall_m',
       'nearest_school_m','nearest_pri_school_m', 
       'nearest_sec_school_m',
       'sec_sch_score_entry_min', 'sec_sch_score_entry_max',
       'nearest_jc_school_m',
       'jc_arts_entry_min', 'jc_arts_entry_max', 'jc_science_entry_min',
       'jc_science_entry_max',
       'mrt_traffic_tap_out_amount_norm', 'mrt_traffic_total_tap_norm',
       'mrt_traffic_weekday_tap_norm', 'mrt_traffic_weekend_tap_norm',
       'mrt_traffic_am_peak_norm', 'mrt_traffic_pm_peak_norm']

cat_col_num = ['MAX_FLOOR',
 'mrt_traffic_busiest_hour',
 'train_lines',
 'station_is_open']
cat_col = [i for i in test.columns if i not in num_col+id_col+[label]+cat_col_num]

In [82]:
# Fit on train only; categories unseen in train are encoded as all zeros in test
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(train[cat_col])

def one_hot_encode(df):
    encoded = pd.DataFrame(ohe.transform(df[cat_col]), columns=ohe.get_feature_names_out(cat_col), index=df.index)
    return pd.concat([df.drop(columns=cat_col), encoded], axis=1)

print(f'Before: Train {train.shape}, Test {test.shape}')
train = one_hot_encode(train)
test = one_hot_encode(test)
ohe_col = list(ohe.get_feature_names_out(cat_col))
print(f'After:  Train {train.shape}, Test {test.shape}')
print(f'{len(cat_col)} categorical features -> {len(ohe_col)} one hot columns')

# Check that train and test have the same encoded columns
assert [c for c in train.columns if c != label] == [c for c in test.columns if c != label]
pd.Series({c: len(ohe.categories_[i]) for i, c in enumerate(cat_col)}, name='n_categories').to_frame()

Before: Train (149999, 75), Test (50000, 74)
After:  Train (149999, 172), Test (50000, 171)
12 categorical features -> 109 one hot columns


,n_categories
FLAT_TYPE,6
FLAT_MODEL,20
dbscan_cluster,48
nearest_school_type,5
nearest_school_nature,3
nearest_school_level,7
pri_sch_type,2
pri_sch_nature,3
sec_sch_type,5
sec_sch_nature,3


In [84]:
train.to_csv(f'{ROOT}/data/features/cleaned/static_features/one_hot/train_one_hot.csv',index = False)
test.to_csv(f'{ROOT}/data/features/cleaned/static_features/one_hot/test_one_hot.csv',index = False)

# Final Comments